
# Getting Started with DeepSpeed ZeRO and Ray Train (Notebook)

This notebook walks through how to combine **DeepSpeed ZeRO** with **Ray Train** to efficiently scale PyTorch training across GPUs and nodes while minimizing memory usage.

It includes:
- A hands-on example of fine-tuning an LLM
- Checkpoint saving and resuming with Ray Train
- Configuring ZeRO for memory and performance (stages, mixed precision, CPU offload)
- Launching a distributed training job

> **Note**: This template is optimized for the Anyscale platform. When running on open-source Ray, you must configure a Ray cluster, install dependencies on all nodes, and set up storage for checkpoints.



## Anyscale-Specific Configuration

On Anyscale, most configuration is automated. When running on open-source Ray, you will need to manually:
- **Configure your Ray Cluster** (multi-node setup, resource allocation)
- **Manage Dependencies** (install on each node)
- **Set Up Storage** (shared or distributed storage for checkpoints)



## Install Dependencies (if needed)

Uncomment and run the cell below if your environment doesn't already have these packages installed.


In [ ]:
%%bash
pip install torch torchvision
pip install transformers datasets==3.6.0 trl
pip install deepspeed ray[train]


## Configuration Constants

We use simple constants instead of `argparse` so this notebook is easier to run. Adjust these as needed.


In [ ]:
import os
os.environ["RAY_TRAIN_V2_ENABLED"] = "1"  # Ensure Ray Train v2 APIs

# ---- Training constants (edit these) ----
MODEL_NAME = "MiniLLM/MiniPLM-Qwen-500M"
BATCH_SIZE = 1
NUM_EPOCHS = 1
SEQ_LENGTH = 512
LEARNING_RATE = 1e-6
ZERO_STAGE = 3

# Ray scaling settings
NUM_WORKERS = 2
USE_GPU = True

# Storage
STORAGE_PATH = "/mnt/cluster_storage/"
EXPERIMENT_PREFIX = "deepspeed_sample"


### 1. Import Packages
We import **Ray Train** for distributed orchestration, **PyTorch** for modeling, **Hugging Face Transformers/Datasets** for models and data, and **DeepSpeed** for ZeRO-based optimization.


In [ ]:

import uuid
import logging
from typing import Dict, Any

import ray
import ray.train
import ray.train.torch
from ray.train.torch import TorchTrainer
from ray.train import ScalingConfig, RunConfig, Checkpoint

import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset, DownloadConfig

import deepspeed

logger = logging.getLogger(__name__)


### 2. Set Up the Dataloader

The function below:

1. Downloads a tokenizer from the Hugging Face Hub (`AutoTokenizer`).  
2. Loads the `ag_news` dataset using Hugging Face’s `load_dataset`.  
3. Applies tokenization with padding and truncation via `map`.  
4. Converts the dataset into a PyTorch `DataLoader`, which handles batching and shuffling.  
5. Finally, use `ray.train.torch.prepare_data_loader` to make the dataloader distributed-ready.


In [ ]:

def get_tokenizer(model_name: str, trust_remote_code: bool = True):
    # (1) Download & configure tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=trust_remote_code)
    if tokenizer.pad_token is None:
        if tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            # Fallback for models without eos_token
            tokenizer.pad_token = tokenizer.convert_ids_to_tokens(2)
    return tokenizer


def setup_dataloader(model_name: str, seq_length: int, batch_size: int) -> DataLoader:
    # (1) Get tokenizer
    tokenizer = get_tokenizer(model_name, trust_remote_code=True)

    # (2) Load dataset
    dataset = load_dataset("ag_news", split="train[:100%]", download_config=DownloadConfig(disable_tqdm=True))

    # (3) Tokenize
    def tokenize_function(examples):
        return tokenizer(examples['text'], padding='max_length', max_length=seq_length, truncation=True)
    tokenized_dataset = dataset.map(tokenize_function, batched=True, num_proc=1, keep_in_memory=True)
    tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask'])

    # (4) Create DataLoader
    data_loader = DataLoader(tokenized_dataset, batch_size=batch_size, shuffle=True)

    # (5) Use prepare_data_loader for distributed training
    return ray.train.torch.prepare_data_loader(data_loader)


> 🚀 **Making the dataloader distributed-ready with Ray**  
> In **data parallelism**, each GPU worker trains on a unique shard of the dataset while holding its own copy of the model; gradients are synchronized after each step.  
> Ray’s `prepare_data_loader` wraps PyTorch’s `DataLoader` and automatically applies a `DistributedSampler`, ensuring workers see disjoint data, splits are balanced, and epoch boundaries are handled correctly.



### 3. Initialize Model and Optimizer

The function below:

1. Loads a pretrained model from the Hugging Face Hub (`AutoModelForCausalLM`).  
2. Defines the optimizer (`AdamW`).  
3. Initializes DeepSpeed with ZeRO options and returns a `DeepSpeedEngine`.


In [ ]:

def setup_model_and_optimizer(model_name: str, learning_rate: float, ds_config: Dict[str, Any]) -> deepspeed.runtime.engine.DeepSpeedEngine:
    # (1) Load pretrained model
    model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

    # (2) Define optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    # (3) Initialize with DeepSpeed (distributed + memory optimizations)
    ds_engine, _, _, _ = deepspeed.initialize(model=model, optimizer=optimizer, config=ds_config)
    return ds_engine



> ⚙️ **Making the model distributed-ready with Ray and DeepSpeed**  
> DeepSpeed’s `initialize` always partitions **optimizer states** (ZeRO Stage 1). Depending on the chosen stage, it can also partition **gradients** (Stage 2) and **model parameters/weights** (Stage 3). This staged approach balances memory savings and communication overhead, and we’ll describe these stages in more detail [later in the tutorial](#deepspeed-zero-stages).



### 4. Checkpointing and Loading

Checkpointing enables fault tolerance and resumability.


In [ ]:

def report_metrics_and_save_checkpoint(
    ds_engine: deepspeed.runtime.engine.DeepSpeedEngine,
    metrics: Dict[str, Any]
) -> None:
    # (1) Create temporary directory
    with tempfile.TemporaryDirectory() as tmp:
        tmp_epoch = os.path.join(tmp, "epoch")
        os.makedirs(tmp_epoch, exist_ok=True)

        # (2) Save checkpoint (partitioned across workers)
        ds_engine.save_checkpoint(tmp_epoch)

        # (3) Synchronize workers
        torch.distributed.barrier()

        # (4) Report metrics and checkpoint to Ray
        ray.train.report(metrics, checkpoint=Checkpoint.from_directory(tmp))


def load_checkpoint(ds_engine: deepspeed.runtime.engine.DeepSpeedEngine, ckpt: ray.train.Checkpoint):
    try:
        # (5) Restore checkpoint into DeepSpeed engine
        with ckpt.as_directory() as checkpoint_dir:
            ds_engine.load_checkpoint(checkpoint_dir)
        torch.distributed.barrier()
    except Exception as e:
        raise RuntimeError(f"Checkpoint loading failed: {e}") from e



> 💾 **Making checkpoints distributed-ready with Ray and DeepSpeed**  
> DeepSpeed saves model and optimizer states in a **partitioned format**, where each worker stores only its shard. This requires synchronization across processes, so all workers must reach the same checkpointing point before proceeding. We use `torch.distributed.barrier()` to ensure that every worker finishes saving before moving on.  
> Finally, `ray.train.report` both reports training metrics and saves the checkpoint to persistent storage, making it accessible for resuming training later.



### 5. Training Iteration

The training loop below runs on each Ray worker (typically one per GPU).


In [ ]:

def train_loop(config: Dict[str, Any]) -> None:
    # (1) Load checkpoint if exists
    ckpt = ray.train.get_checkpoint()
    if ckpt:
        load_checkpoint(ds_engine, ckpt)  # ds_engine will be defined after initialization below

    # (2) Set up dataloader
    train_loader = setup_dataloader(config["model_name"], config["seq_length"], config["batch_size"])

    # (3) Initialize model + optimizer with DeepSpeed
    ds_engine_local = setup_model_and_optimizer(config["model_name"], config["learning_rate"], config["ds_config"])

    # (4) Get device for this worker
    device = ray.train.torch.get_device()

    for epoch in range(config["epochs"]):
        # (6) Ensure unique shard per worker when using multiple GPUs
        if ray.train.get_context().get_world_size() > 1 and hasattr(train_loader, "sampler"):
            sampler = getattr(train_loader, "sampler", None)
            if sampler and hasattr(sampler, "set_epoch"):
                sampler.set_epoch(epoch)

        running_loss = 0.0
        num_batches = 0

        # (7) Iterate over batches
        for step, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            # Forward pass
            outputs = ds_engine_local(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=input_ids,
                use_cache=False
            )
            loss = outputs.loss
            print(f"step {step} loss: {loss.item()}")

            # Backward pass + optimizer step
            ds_engine_local.backward(loss)
            ds_engine_local.step()

            running_loss += loss.item()
            num_batches += 1

        # (8) Report metrics + save checkpoint
        report_metrics_and_save_checkpoint(ds_engine_local, {"loss": running_loss / num_batches, "epoch": epoch})



> 💡 **Coordinating distributed training with Ray and DeepSpeed**  
> Ray launches this `train_loop` on each worker, while DeepSpeed manages partitioning and memory optimizations. With **data parallelism**, each worker processes a unique shard of data, computes gradients locally, and participates in synchronization so parameters stay in sync.



### 6. Configure DeepSpeed and Launch Trainer

We assemble the Ray scaling config, DeepSpeed config, and training loop config, then launch training with `TorchTrainer`.


In [ ]:

# Ray scaling configuration
scaling_config = ScalingConfig(num_workers=NUM_WORKERS, use_gpu=USE_GPU)

# DeepSpeed configuration
ds_config = {
    "train_micro_batch_size_per_gpu": BATCH_SIZE,
    "bf16": {"enabled": True},
    "grad_accum_dtype": "bf16",
    "zero_optimization": {
        "stage": ZERO_STAGE,
        "overlap_comm": True,
        "contiguous_gradients": True,
    },
    "gradient_clipping": 1.0,
}

# Training loop configuration
train_loop_config = {
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "ds_config": ds_config,
    "model_name": MODEL_NAME,
    "seq_length": SEQ_LENGTH,
}

# Ray run configuration
run_config = RunConfig(
    storage_path=STORAGE_PATH,
    name=f"{EXPERIMENT_PREFIX}_{uuid.uuid4().hex[:8]}",
)

# Create and launch the trainer
trainer = TorchTrainer(
    train_loop_per_worker=train_loop,
    scaling_config=scaling_config,
    train_loop_config=train_loop_config,
    run_config=run_config,
)

# To actually run training, execute the following:
result = trainer.fit()
print(f"Training finished. Result: {result}")



## Advanced Configurations

### DeepSpeed ZeRO Stages
- **Stage 1**: Partitions optimizer states (always on when using ZeRO).  
- **Stage 2**: Additionally partitions gradients.  
- **Stage 3**: Additionally partitions model parameters/weights.

You can select the stage via `ds_config["zero_optimization"]["stage"]`. See the DeepSpeed docs for more details.

### Mixed Precision
Enable BF16 or FP16:
```python
ds_config = {
    "bf16": {"enabled": True},  # or "fp16": {"enabled": True}
}
```

### CPU Offloading
Reduce GPU memory pressure by offloading to CPU (at the cost of PCIe traffic):
```python
ds_config = {
    "offload_param": {"device": "cpu", "pin_memory": True},
    # or
    "offload_optimizer": {"device": "cpu", "pin_memory": True},
}
```
